# 03. Motor Predictivo de Tarifas: Machine Learning + SHAP

**Entrada:** `../data/processed/listings_features.parquet`  
**Modelo final:** `../models/xgboost_model_final.pkl`

## Objetivos

1. Definir `price` como variable objetivo.
2. Crear un **holdout espacial aproximado 80/20**, formado por celdas H3 completas y territorialmente equilibrado.
3. Garantizar que ninguna celda `h3_res8` aparezca simultáneamente en train y test.
4. Aplicar validación cruzada espacial con `GroupKFold` utilizando también `h3_res8` como grupo.
5. Construir pipelines reproducibles para Dummy, LightGBM, XGBoost y CatBoost.
6. Crear medias de precio H3 de forma segura, aprendidas únicamente con los datos de entrenamiento de cada fold.
7. Comparar modelos mediante R², RMSE, MAE y MAPE.
8. Ajustar LightGBM y XGBoost mediante Optuna.
9. Guardar el pipeline completo y generar explicaciones SHAP.

> El holdout y la validación cruzada utilizan grupos `h3_res8` completos. `h3_res9` se mantiene como predictor para el target encoding, pero no define los folds.


## 0. Instalación de dependencias

In [ ]:
# Descomenta solamente si es necesario y reinicia después el kernel.
%pip install lightgbm xgboost catboost optuna shap joblib scikit-learn pyarrow

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.utils import check_random_state
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import optuna
import shap
import joblib

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports/modeling")
FIGURES_DIR = REPORTS_DIR / "figures"

INPUT_FEATURES = (
    PROCESSED_DIR
    / "listings_features.parquet"
)

OUTPUT_XGBOOST_MODEL = (
    MODELS_DIR
    / "xgboost_model_final.pkl"
)


OUTPUT_METRICS = (
    REPORTS_DIR
    / "xgboost_final_test_metrics.csv"
)

OUTPUT_BEST_PARAMS = (
    REPORTS_DIR
    / "xgboost_best_params.json"
)

OUTPUT_TEST_PREDICTIONS = (
    REPORTS_DIR
    / "xgboost_test_predictions.csv"
)

OUTPUT_SHAP_BEESWARM = (
    FIGURES_DIR
    / "shap_beeswarm.png"
)

OUTPUT_SHAP_WATERFALL = (
    FIGURES_DIR
    / "shap_waterfall.png"
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# PARÁMETROS
# ------------------------------------------------------------

RANDOM_STATE = 42
TARGET_COLUMN = "price"

HOLDOUT_TEST_RATIO = 0.20
N_SPATIAL_FOLDS = 5

HOLDOUT_SEARCH_TRIALS = 5000
HOLDOUT_RATIO_TOLERANCE = 0.01

# Número de trials utilizado en la optimización con Optuna.
N_OPTUNA_TRIALS = 25

# Muestra máxima utilizada en SHAP para evitar un coste excesivo.
MAX_SHAP_ROWS = 1000


print(f"Entrada: {INPUT_FEATURES}")
print(f"Modelo final: {OUTPUT_XGBOOST_MODEL}")

## 2. Versiones de las librerías

In [ ]:
import sklearn
import lightgbm
import xgboost
import catboost

print("scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)
print("XGBoost:", xgboost.__version__)
print("CatBoost:", catboost.__version__)
print("Optuna:", optuna.__version__)
print("SHAP:", shap.__version__)

## 3. Carga del dataset enriquecido

In [ ]:
if not INPUT_FEATURES.exists():
    raise FileNotFoundError(
        f"No existe {INPUT_FEATURES}. "
        "Ejecuta primero la Fase 2."
    )

df = pd.read_parquet(INPUT_FEATURES)

print(
    f"Dataset cargado: {df.shape[0]:,} filas × "
    f"{df.shape[1]} columnas"
)

display(df.head())

## 4. Validación de `price`

In [ ]:
if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"No existe la variable objetivo {TARGET_COLUMN!r}."
    )

df[TARGET_COLUMN] = pd.to_numeric(
    df[TARGET_COLUMN],
    errors="coerce",
)

target_summary = pd.DataFrame({
    "metrica": [
        "Filas",
        "Nulos",
        "Valores <= 0",
        "Mínimo",
        "Percentil 25",
        "Mediana",
        "Media",
        "Percentil 75",
        "Percentil 95",
        "Máximo",
    ],
    "valor": [
        len(df),
        int(df[TARGET_COLUMN].isna().sum()),
        int((df[TARGET_COLUMN] <= 0).sum()),
        df[TARGET_COLUMN].min(),
        df[TARGET_COLUMN].quantile(0.25),
        df[TARGET_COLUMN].median(),
        df[TARGET_COLUMN].mean(),
        df[TARGET_COLUMN].quantile(0.75),
        df[TARGET_COLUMN].quantile(0.95),
        df[TARGET_COLUMN].max(),
    ],
})

display(target_summary)

rows_before = len(df)

df = (
    df.loc[
        df[TARGET_COLUMN].notna()
        &
        (df[TARGET_COLUMN] > 0)
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "Filas eliminadas por price inválido: "
    f"{rows_before - len(df):,}"
)

## 5. Selección de estratos y grupos espaciales

- La variable territorial se utiliza para buscar un holdout espacial equilibrado.
- `h3_res8` separa train y test mediante grupos completos.
- `h3_res8` se utiliza también para la validación cruzada espacial.
- `h3_res9` permanece en las variables de entrada para el target encoding, pero no define los folds.


In [ ]:
possible_strata_columns = [
    "neighbourhood_group",
    "district",
    "neighbourhood_cleansed",
]

STRATA_COLUMN = next(
    (
        column
        for column in possible_strata_columns
        if column in df.columns
    ),
    None,
)

if STRATA_COLUMN is None:
    raise KeyError(
        "No se encontró una columna territorial "
        "para la estratificación."
    )

required_spatial_columns = [
    "h3_res8",
    "h3_res9",
]

missing_spatial_columns = [
    column
    for column in required_spatial_columns
    if column not in df.columns
]

if missing_spatial_columns:
    raise KeyError(
        "Faltan columnas espaciales: "
        f"{missing_spatial_columns}"
    )

df[STRATA_COLUMN] = (
    df[STRATA_COLUMN]
    .astype("string")
    .fillna("Desconocido")
)

for column in required_spatial_columns:
    df[column] = (
        df[column]
        .astype("string")
        .fillna("H3_desconocido")
    )

print("Estrato territorial:", STRATA_COLUMN)
print(
    "Categorías territoriales:",
    df[STRATA_COLUMN].nunique(),
)
print("Grupos H3 res8:", df["h3_res8"].nunique())
print("Grupos H3 res9:", df["h3_res9"].nunique())

## 6. Definición inicial de X e y

In [ ]:
# Los índices H3 se conservan en X porque el transformer de target
# encoding los necesita. Después los elimina antes del modelo.

identifier_columns = [
    "id",
    "listing_id",
    "listing_rag_id",
]

existing_identifier_columns = [
    column
    for column in identifier_columns
    if column in df.columns
]

X = df.drop(
    columns=[
        TARGET_COLUMN,
        *existing_identifier_columns,
    ],
    errors="ignore",
).copy()

y = df[TARGET_COLUMN].copy()

strata = df[STRATA_COLUMN].copy()
groups_res8 = df["h3_res8"].copy()
groups_res9 = df["h3_res9"].copy()

print(
    f"X inicial: {X.shape[0]:,} filas × "
    f"{X.shape[1]} columnas"
)
print(f"y: {len(y):,} valores")

## 7. Holdout espacial 80/20 territorialmente equilibrado

Se generan múltiples particiones candidatas de celdas `h3_res8` completas. Se priorizan las que dejan un test cercano al 20 % y, entre ellas, se selecciona la de menor desequilibrio territorial.

No se utiliza `price` para escoger la partición.


In [ ]:
rng = check_random_state(RANDOM_STATE)

cell_counts = groups_res8.value_counts().sort_index()
all_cells = cell_counts.index.to_numpy()
total_rows = len(df)

territory_levels = (
    strata.astype("string")
    .fillna("Desconocido")
    .unique()
    .tolist()
)

candidate_rows = []

for trial in range(HOLDOUT_SEARCH_TRIALS):
    shuffled_cells = all_cells.copy()
    rng.shuffle(shuffled_cells)

    cumulative_rows = np.cumsum(
        [int(cell_counts.loc[cell]) for cell in shuffled_cells]
    )

    cumulative_ratios = cumulative_rows / total_rows

    cut_position = int(
        np.argmin(
            np.abs(cumulative_ratios - HOLDOUT_TEST_RATIO)
        )
    )

    candidate_test_cells = set(
        shuffled_cells[: cut_position + 1]
    )

    candidate_test_mask = groups_res8.isin(
        candidate_test_cells
    )

    candidate_train_mask = ~candidate_test_mask

    candidate_test_ratio = float(
        candidate_test_mask.mean()
    )

    train_distribution = (
        strata.loc[candidate_train_mask]
        .value_counts(normalize=True)
        .reindex(territory_levels, fill_value=0.0)
    )

    test_distribution = (
        strata.loc[candidate_test_mask]
        .value_counts(normalize=True)
        .reindex(territory_levels, fill_value=0.0)
    )

    distribution_difference = (
        train_distribution - test_distribution
    ).abs()

    candidate_rows.append(
        {
            "trial": trial,
            "test_ratio": candidate_test_ratio,
            "ratio_error": abs(
                candidate_test_ratio - HOLDOUT_TEST_RATIO
            ),
            "territorial_mean_difference": float(
                distribution_difference.mean()
            ),
            "territorial_max_difference": float(
                distribution_difference.max()
            ),
            "test_cells": candidate_test_cells,
        }
    )

holdout_candidates = pd.DataFrame(candidate_rows)

eligible_candidates = holdout_candidates.loc[
    holdout_candidates["ratio_error"]
    <= HOLDOUT_RATIO_TOLERANCE
].copy()

if eligible_candidates.empty:
    minimum_ratio_error = (
        holdout_candidates["ratio_error"].min()
    )

    eligible_candidates = holdout_candidates.loc[
        holdout_candidates["ratio_error"]
        == minimum_ratio_error
    ].copy()

eligible_candidates["territorial_score"] = (
    eligible_candidates["territorial_mean_difference"]
    + 0.25
    * eligible_candidates["territorial_max_difference"]
)

best_candidate = (
    eligible_candidates
    .sort_values(
        ["territorial_score", "ratio_error", "trial"]
    )
    .iloc[0]
)

test_cells = set(best_candidate["test_cells"])

test_mask = groups_res8.isin(test_cells)

train_indices = np.where(~test_mask)[0]
test_indices = np.where(test_mask)[0]

X_train = X.iloc[train_indices].reset_index(drop=True)
X_test = X.iloc[test_indices].reset_index(drop=True)

y_train = y.iloc[train_indices].reset_index(drop=True)
y_test = y.iloc[test_indices].reset_index(drop=True)

strata_train = strata.iloc[train_indices].reset_index(drop=True)
strata_test = strata.iloc[test_indices].reset_index(drop=True)

groups_train_res8 = (
    groups_res8.iloc[train_indices]
    .reset_index(drop=True)
)

groups_test_res8 = (
    groups_res8.iloc[test_indices]
    .reset_index(drop=True)
)

print(f"Candidaturas evaluadas: {HOLDOUT_SEARCH_TRIALS:,}")
print(
    f"Celdas H3 res8 en test: "
    f"{len(test_cells)} de {len(cell_counts)}"
)
print(
    f"Train: {len(X_train):,} "
    f"({len(X_train) / len(X):.2%})"
)
print(
    f"Test: {len(X_test):,} "
    f"({len(X_test) / len(X):.2%})"
)
print(
    "Diferencia territorial media candidata:",
    round(
        float(
            best_candidate[
                "territorial_mean_difference"
            ]
        ),
        4,
    ),
)
print(
    "Diferencia territorial máxima candidata:",
    round(
        float(
            best_candidate[
                "territorial_max_difference"
            ]
        ),
        4,
    ),
)


## 8. Comprobación de ausencia de solapamiento espacial

In [ ]:
train_groups = set(groups_train_res8.unique())
test_groups = set(groups_test_res8.unique())

overlap = train_groups.intersection(test_groups)

print(
    "Celdas H3 res8 compartidas entre train y test:",
    len(overlap),
)

assert len(overlap) == 0

print("No existe solapamiento espacial H3 res8.")

## 9. Auditoría de la distribución territorial

Se compara la proporción de cada territorio en train y test para verificar que la partición mantiene un equilibrio razonable sin perder independencia espacial.


In [ ]:
territorial_distribution = pd.concat(
    [
        strata_train
        .value_counts(normalize=True)
        .rename("train"),
        strata_test
        .value_counts(normalize=True)
        .rename("test"),
    ],
    axis=1,
).fillna(0)

territorial_distribution["difference_absolute"] = (
    territorial_distribution["train"]
    - territorial_distribution["test"]
).abs()

territorial_distribution = (
    territorial_distribution
    .sort_values(
        "difference_absolute",
        ascending=False,
    )
)

display(territorial_distribution)

mean_territorial_difference = float(
    territorial_distribution[
        "difference_absolute"
    ].mean()
)

max_territorial_difference = float(
    territorial_distribution[
        "difference_absolute"
    ].max()
)

print(
    "Diferencia territorial media:",
    round(mean_territorial_difference, 4),
)

print(
    "Diferencia territorial máxima:",
    round(max_territorial_difference, 4),
)

print(
    "Ratio final de test:",
    f"{len(X_test) / len(X):.2%}",
)


## 10. Target encoding H3 sin fuga de información

Este transformer aprende, durante cada `fit`, la media de `price` por celda H3 utilizando exclusivamente el subconjunto de entrenamiento recibido.

Para celdas desconocidas utiliza la media global de train. Se aplica suavizado para reducir la inestabilidad de celdas con pocos alojamientos.

In [ ]:
class H3TargetMeanEncoder(
    BaseEstimator,
    TransformerMixin,
):
    def __init__(
        self,
        h3_columns=("h3_res8", "h3_res9"),
        smoothing=20.0,
        drop_original=True,
    ):
        self.h3_columns = h3_columns
        self.smoothing = smoothing
        self.drop_original = drop_original

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)

        self.global_mean_ = float(y.mean())
        self.encoding_maps_ = {}

        for column in self.h3_columns:
            if column not in X.columns:
                continue

            temporary = pd.DataFrame({
                "group": (
                    X[column]
                    .astype("string")
                    .fillna("H3_desconocido")
                ),
                "target": y,
            })

            stats = (
                temporary
                .groupby("group")["target"]
                .agg(["mean", "count"])
            )

            smoothed_mean = (
                stats["count"] * stats["mean"]
                +
                self.smoothing * self.global_mean_
            ) / (
                stats["count"]
                +
                self.smoothing
            )

            self.encoding_maps_[column] = (
                smoothed_mean.to_dict()
            )

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for column in self.h3_columns:
            if column not in X.columns:
                continue

            encoded_column = (
                f"{column}_mean_price_te"
            )

            X[encoded_column] = (
                X[column]
                .astype("string")
                .fillna("H3_desconocido")
                .map(
                    self.encoding_maps_.get(
                        column,
                        {},
                    )
                )
                .fillna(self.global_mean_)
                .astype("float64")
            )

        if self.drop_original:
            X = X.drop(
                columns=[
                    column
                    for column in self.h3_columns
                    if column in X.columns
                ],
                errors="ignore",
            )

        return X

## 11. Detección de columnas después del target encoding

In [ ]:
schema_encoder = H3TargetMeanEncoder(
    smoothing=20.0,
)

X_train_schema = schema_encoder.fit_transform(
    X_train,
    y_train,
)

numeric_columns = (
    X_train_schema
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_columns = (
    X_train_schema
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
            "bool",
        ]
    )
    .columns
    .tolist()
)

unsupported_columns = [
    column
    for column in X_train_schema.columns
    if column not in (
        numeric_columns
        + categorical_columns
    )
]

print("Numéricas:", len(numeric_columns))
print("Categóricas:", len(categorical_columns))
print("Otros tipos:", unsupported_columns)

if unsupported_columns:
    raise TypeError(
        "Existen columnas con tipos no contemplados: "
        f"{unsupported_columns}"
    )

print("\nVariables categóricas:")
print(categorical_columns)

## 12. Preprocesamiento de LightGBM y XGBoost

- Numéricas: imputación por mediana.
- Categóricas: imputación por moda + OneHotEncoder.
- `handle_unknown="ignore"` evita errores con categorías nuevas.
- `min_frequency=10` agrupa categorías poco frecuentes.
- No se aplica escalado porque los modelos de árboles no lo necesitan.
- `VarianceThreshold` elimina columnas constantes aprendidas únicamente con train.

In [ ]:
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True,
    )

except TypeError:
    # Compatibilidad con versiones antiguas de scikit-learn.
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True,
    )

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "onehot",
            one_hot_encoder,
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_columns,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_columns,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

## 13. Funciones de evaluación

In [ ]:
def regression_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    r2 = r2_score(
        y_true,
        y_pred,
    )

    nonzero_mask = y_true != 0

    if nonzero_mask.any():
        mape = (
            mean_absolute_percentage_error(
                y_true[nonzero_mask],
                y_pred[nonzero_mask],
            )
            * 100
        )
    else:
        mape = np.nan

    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "MAPE_percent": float(mape),
    }


spatial_cv = GroupKFold(
    n_splits=N_SPATIAL_FOLDS,
)

cv_splits = list(
    spatial_cv.split(
        X_train,
        y_train,
        groups=groups_train_res8,
    )
)

print(
    "Folds espaciales preparados:",
    len(cv_splits),
)

for fold_number, (
    fold_train_indices,
    fold_valid_indices,
) in enumerate(
    cv_splits,
    start=1,
):
    fold_train_groups = set(
        groups_train_res8
        .iloc[fold_train_indices]
        .unique()
    )

    fold_valid_groups = set(
        groups_train_res8
        .iloc[fold_valid_indices]
        .unique()
    )

    overlap = fold_train_groups.intersection(
        fold_valid_groups
    )

    assert len(overlap) == 0

    print(
        f"Fold {fold_number}: "
        f"{len(fold_train_indices):,} train · "
        f"{len(fold_valid_indices):,} valid · "
        f"solapamiento H3 res8 = {len(overlap)}"
    )

print(
    "alidación cruzada espacial "
    "sin solapamiento H3 res8."
)


In [ ]:
def evaluate_sklearn_pipeline_cv(
    model_name,
    pipeline,
    X_data,
    y_data,
    splits,
):
    fold_rows = []

    start_time = time.time()

    for fold_number, (
        fold_train_indices,
        fold_valid_indices,
    ) in enumerate(
        splits,
        start=1,
    ):
        fold_pipeline = clone(pipeline)

        X_fold_train = (
            X_data
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        X_fold_valid = (
            X_data
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        y_fold_train = (
            y_data
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        y_fold_valid = (
            y_data
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_predictions = fold_pipeline.predict(
            X_fold_valid
        )

        metrics = regression_metrics(
            y_fold_valid,
            fold_predictions,
        )

        metrics["model"] = model_name
        metrics["fold"] = fold_number

        fold_rows.append(metrics)

        print(
            f"{model_name} | fold {fold_number} | "
            f"MAE={metrics['MAE']:.3f} | "
            f"RMSE={metrics['RMSE']:.3f} | "
            f"R2={metrics['R2']:.3f}"
        )

    elapsed_seconds = time.time() - start_time

    fold_results = pd.DataFrame(
        fold_rows
    )

    summary = {
        "model": model_name,
        "CV_R2_mean": fold_results["R2"].mean(),
        "CV_R2_std": fold_results["R2"].std(),
        "CV_RMSE_mean": fold_results["RMSE"].mean(),
        "CV_RMSE_std": fold_results["RMSE"].std(),
        "CV_MAE_mean": fold_results["MAE"].mean(),
        "CV_MAE_std": fold_results["MAE"].std(),
        "CV_MAPE_mean": (
            fold_results["MAPE_percent"].mean()
        ),
        "training_seconds": elapsed_seconds,
    }

    return summary, fold_results

## 14. Pipelines baseline

In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        (
            "h3_target_encoder",
            H3TargetMeanEncoder(
                smoothing=20.0,
            ),
        ),
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "variance_filter",
            VarianceThreshold(
                threshold=0.0,
            ),
        ),
        (
            "model",
            DummyRegressor(
                strategy="median",
            ),
        ),
    ]
)

lightgbm_pipeline = Pipeline(
    steps=[
        (
            "h3_target_encoder",
            H3TargetMeanEncoder(
                smoothing=20.0,
            ),
        ),
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "variance_filter",
            VarianceThreshold(
                threshold=0.0,
            ),
        ),
        (
            "model",
            LGBMRegressor(
                objective="regression_l1",
                n_estimators=700,
                learning_rate=0.04,
                num_leaves=31,
                max_depth=-1,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.0,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        ),
    ]
)

xgboost_pipeline = Pipeline(
    steps=[
        (
            "h3_target_encoder",
            H3TargetMeanEncoder(
                smoothing=20.0,
            ),
        ),
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "variance_filter",
            VarianceThreshold(
                threshold=0.0,
            ),
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                n_estimators=700,
                learning_rate=0.04,
                max_depth=7,
                min_child_weight=3,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.0,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ),
        ),
    ]
)

## 15. Evaluación baseline de los primeros modelos

In [ ]:
baseline_summaries = []
baseline_fold_results = []

for model_name, pipeline in [
    ("DummyRegressor", dummy_pipeline),
    ("LightGBM_baseline", lightgbm_pipeline),
    ("XGBoost_baseline", xgboost_pipeline),
]:
    summary, folds = (
        evaluate_sklearn_pipeline_cv(
            model_name=model_name,
            pipeline=pipeline,
            X_data=X_train,
            y_data=y_train,
            splits=cv_splits,
        )
    )

    baseline_summaries.append(summary)
    baseline_fold_results.append(folds)

baseline_comparison = pd.DataFrame(
    baseline_summaries
).sort_values(
    "CV_MAE_mean"
)

display(baseline_comparison)

## 16. Preparación específica de CatBoost

CatBoost utiliza variables categóricas nativas. Por tanto:

- no se aplica OneHotEncoder;
- las categóricas se imputan y convierten a texto;
- las numéricas se imputan por la mediana aprendida en cada fold;
- el target encoding H3 sigue ajustándose solamente con el train de cada fold.

In [ ]:
def prepare_catboost_data(
    X_fit,
    y_fit,
    X_transform,
    smoothing=20.0,
):
    h3_encoder = H3TargetMeanEncoder(
        smoothing=smoothing,
    )

    X_fit_encoded = (
        h3_encoder
        .fit_transform(
            X_fit,
            y_fit,
        )
        .reset_index(drop=True)
    )

    X_transform_encoded = (
        h3_encoder
        .transform(X_transform)
        .reset_index(drop=True)
    )

    categorical_columns_cb = (
        X_fit_encoded
        .select_dtypes(
            include=[
                "object",
                "string",
                "category",
                "bool",
            ]
        )
        .columns
        .tolist()
    )

    numeric_columns_cb = (
        X_fit_encoded
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    # Imputación numérica aprendida en train.
    numeric_medians = (
        X_fit_encoded[numeric_columns_cb]
        .median()
    )

    X_fit_encoded[numeric_columns_cb] = (
        X_fit_encoded[numeric_columns_cb]
        .fillna(numeric_medians)
    )

    X_transform_encoded[numeric_columns_cb] = (
        X_transform_encoded[numeric_columns_cb]
        .fillna(numeric_medians)
    )

    # Imputación categórica.
    for column in categorical_columns_cb:
        X_fit_encoded[column] = (
            X_fit_encoded[column]
            .astype("string")
            .fillna("Desconocido")
            .astype(str)
        )

        X_transform_encoded[column] = (
            X_transform_encoded[column]
            .astype("string")
            .fillna("Desconocido")
            .astype(str)
        )

    X_transform_encoded = (
        X_transform_encoded[
            X_fit_encoded.columns
        ]
    )

    categorical_indices = [
        X_fit_encoded.columns.get_loc(column)
        for column in categorical_columns_cb
    ]

    return (
        X_fit_encoded,
        X_transform_encoded,
        categorical_indices,
        h3_encoder,
    )

In [ ]:
def evaluate_catboost_cv(
    X_data,
    y_data,
    splits,
):
    fold_rows = []
    start_time = time.time()

    for fold_number, (
        fold_train_indices,
        fold_valid_indices,
    ) in enumerate(
        splits,
        start=1,
    ):
        X_fold_train = (
            X_data
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        X_fold_valid = (
            X_data
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        y_fold_train = (
            y_data
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        y_fold_valid = (
            y_data
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        (
            X_cb_train,
            X_cb_valid,
            categorical_indices,
            _,
        ) = prepare_catboost_data(
            X_fit=X_fold_train,
            y_fit=y_fold_train,
            X_transform=X_fold_valid,
            smoothing=20.0,
        )

        model = CatBoostRegressor(
            loss_function="MAE",
            iterations=700,
            learning_rate=0.04,
            depth=8,
            l2_leaf_reg=3.0,
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )

        model.fit(
            X_cb_train,
            y_fold_train,
            cat_features=categorical_indices,
        )

        predictions = model.predict(
            X_cb_valid
        )

        metrics = regression_metrics(
            y_fold_valid,
            predictions,
        )

        metrics["model"] = "CatBoost_baseline"
        metrics["fold"] = fold_number
        fold_rows.append(metrics)

        print(
            f"CatBoost | fold {fold_number} | "
            f"MAE={metrics['MAE']:.3f} | "
            f"RMSE={metrics['RMSE']:.3f} | "
            f"R2={metrics['R2']:.3f}"
        )

    elapsed_seconds = time.time() - start_time

    fold_results = pd.DataFrame(
        fold_rows
    )

    summary = {
        "model": "CatBoost_baseline",
        "CV_R2_mean": fold_results["R2"].mean(),
        "CV_R2_std": fold_results["R2"].std(),
        "CV_RMSE_mean": fold_results["RMSE"].mean(),
        "CV_RMSE_std": fold_results["RMSE"].std(),
        "CV_MAE_mean": fold_results["MAE"].mean(),
        "CV_MAE_std": fold_results["MAE"].std(),
        "CV_MAPE_mean": (
            fold_results["MAPE_percent"].mean()
        ),
        "training_seconds": elapsed_seconds,
    }

    return summary, fold_results

## 17. Evaluación baseline de CatBoost

In [ ]:
catboost_summary, catboost_folds = (
    evaluate_catboost_cv(
        X_data=X_train,
        y_data=y_train,
        splits=cv_splits,
    )
)

all_baseline_summaries = (
    baseline_summaries
    + [catboost_summary]
)

model_comparison_baseline = (
    pd.DataFrame(
        all_baseline_summaries
    )
    .sort_values("CV_MAE_mean")
    .reset_index(drop=True)
)

display(model_comparison_baseline)

## 18. Ajuste de LightGBM y XGBoost con Optuna

Tras la evaluación baseline, se seleccionan LightGBM y XGBoost para la fase de
optimización. Ambos ofrecen un buen equilibrio entre rendimiento predictivo y
coste computacional, mientras que CatBoost presenta un tiempo de entrenamiento
considerablemente mayor.

Los hiperparámetros de LightGBM y XGBoost se optimizan mediante Optuna utilizando
la misma validación cruzada espacial definida anteriormente.

La función objetivo minimiza el MAE medio de validación cruzada espacial. En cada trial se vuelven a ajustar dentro del fold:

1. el target encoding H3;
2. la imputación;
3. el OneHotEncoder;
4. el filtro de varianza;
5. el modelo correspondiente.

De esta forma no se utiliza información del conjunto de validación durante el entrenamiento.

In [ ]:
def optuna_objective_lightgbm(trial):

    model = LGBMRegressor(
        objective="regression_l1",
        n_estimators=trial.suggest_int(
            "n_estimators",
            300,
            1400,
            step=100,
        ),
        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.12,
            log=True,
        ),
        num_leaves=trial.suggest_int(
            "num_leaves",
            15,
            127,
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            4,
            14,
        ),
        min_child_samples=trial.suggest_int(
            "min_child_samples",
            10,
            100,
        ),
        subsample=trial.suggest_float(
            "subsample",
            0.65,
            1.0,
        ),
        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.65,
            1.0,
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha",
            1e-4,
            10.0,
            log=True,
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda",
            1e-4,
            10.0,
            log=True,
        ),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )

    pipeline = Pipeline(
        steps=[
            (
                "h3_target_encoder",
                H3TargetMeanEncoder(
                    smoothing=trial.suggest_float(
                        "h3_smoothing",
                        5.0,
                        100.0,
                        log=True,
                    ),
                ),
            ),
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "variance_filter",
                VarianceThreshold(
                    threshold=0.0,
                ),
            ),
            (
                "model",
                model,
            ),
        ]
    )

    fold_mae = []

    for (
        fold_train_indices,
        fold_valid_indices,
    ) in cv_splits:

        fold_pipeline = clone(
            pipeline
        )

        X_fold_train = (
            X_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        X_fold_valid = (
            X_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        y_fold_train = (
            y_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        y_fold_valid = (
            y_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_predictions = (
            fold_pipeline
            .predict(X_fold_valid)
        )

        fold_mae.append(
            mean_absolute_error(
                y_fold_valid,
                fold_predictions,
            )
        )

    return float(
        np.mean(fold_mae)
    )


def optuna_objective_xgboost(trial):

    model = XGBRegressor(
        objective="reg:absoluteerror",

        n_estimators=trial.suggest_int(
            "n_estimators",
            300,
            1400,
            step=100,
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.12,
            log=True,
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            12,
        ),

        min_child_weight=trial.suggest_float(
            "min_child_weight",
            1.0,
            15.0,
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.65,
            1.0,
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.65,
            1.0,
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            1e-4,
            10.0,
            log=True,
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            1e-4,
            10.0,
            log=True,
        ),

        gamma=trial.suggest_float(
            "gamma",
            1e-4,
            5.0,
            log=True,
        ),

        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
    )

    pipeline = Pipeline(
        steps=[
            (
                "h3_target_encoder",
                H3TargetMeanEncoder(
                    smoothing=trial.suggest_float(
                        "h3_smoothing",
                        5.0,
                        100.0,
                        log=True,
                    ),
                ),
            ),
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "variance_filter",
                VarianceThreshold(
                    threshold=0.0,
                ),
            ),
            (
                "model",
                model,
            ),
        ]
    )

    fold_mae = []

    for (
        fold_train_indices,
        fold_valid_indices,
    ) in cv_splits:

        fold_pipeline = clone(
            pipeline
        )

        X_fold_train = (
            X_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        X_fold_valid = (
            X_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        y_fold_train = (
            y_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        y_fold_valid = (
            y_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_predictions = (
            fold_pipeline
            .predict(X_fold_valid)
        )

        fold_mae.append(
            mean_absolute_error(
                y_fold_valid,
                fold_predictions,
            )
        )

    return float(
        np.mean(fold_mae)
    )

In [ ]:
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

# ============================================================
# LIGHTGBM
# ============================================================

study_lightgbm = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
    ),
)

study_lightgbm.optimize(
    optuna_objective_lightgbm,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
)

print("=" * 70)
print("LIGHTGBM")
print("=" * 70)

print(
    "Mejor MAE CV:",
    study_lightgbm.best_value,
)

display(
    pd.DataFrame(
        study_lightgbm.best_params.items(),
        columns=[
            "parametro",
            "valor",
        ],
    )
)


# ============================================================
# XGBOOST
# ============================================================

study_xgboost = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
    ),
)

study_xgboost.optimize(
    optuna_objective_xgboost,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
)

print()
print("=" * 70)
print("XGBOOST")
print("=" * 70)

print(
    "Mejor MAE CV:",
    study_xgboost.best_value,
)

display(
    pd.DataFrame(
        study_xgboost.best_params.items(),
        columns=[
            "parametro",
            "valor",
        ],
    )
)

## 19. Experimentos sobre la transformación de la variable objetivo

Tras la optimización de LightGBM y XGBoost mediante validación cruzada espacial,
XGBoost presenta el mejor resultado medio de MAE entre los modelos ajustados.
A partir de este modelo se comparan tres estrategias para representar la variable objetivo:

- **A — Precio directo con pérdida absoluta:** XGBoost entrenado directamente sobre `price` utilizando `reg:absoluteerror`.
- **B — Precio directo con pérdida cuadrática:** XGBoost entrenado directamente sobre `price` utilizando `reg:squarederror`.
- **C — Transformación logarítmica:** XGBoost entrenado sobre `log1p(price)` utilizando `reg:squarederror`, transformando posteriormente las predicciones mediante `expm1`.

Las tres alternativas utilizan exactamente los mismos folds espaciales `h3_res8`, las mismas variables predictoras y el mismo esquema de preprocesamiento.

La selección se realiza exclusivamente a partir de la validación cruzada espacial sobre el conjunto de entrenamiento. El holdout del 20 % no interviene en esta decisión.

In [ ]:
def evaluate_xgboost_target_experiment(
    experiment_name,
    objective,
    transform_target=False,
):

    fold_results = []

    for fold_number, (
        fold_train_indices,
        fold_valid_indices,
    ) in enumerate(
        cv_splits,
        start=1,
    ):

        X_fold_train = (
            X_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        X_fold_valid = (
            X_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        y_fold_train_original = (
            y_train
            .iloc[fold_train_indices]
            .reset_index(drop=True)
        )

        y_fold_valid_original = (
            y_train
            .iloc[fold_valid_indices]
            .reset_index(drop=True)
        )

        if transform_target:
            y_fold_train_model = np.log1p(
                y_fold_train_original
            )
        else:
            y_fold_train_model = (
                y_fold_train_original.copy()
            )

        best_params = (
            study_xgboost
            .best_params
            .copy()
        )

        h3_smoothing = best_params.pop(
            "h3_smoothing"
        )

        model = XGBRegressor(
            objective=objective,
            **best_params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )

        pipeline = Pipeline(
            steps=[
                (
                    "h3_target_encoder",
                    H3TargetMeanEncoder(
                        smoothing=h3_smoothing,
                    ),
                ),
                (
                    "preprocessor",
                    clone(preprocessor),
                ),
                (
                    "variance_filter",
                    VarianceThreshold(
                        threshold=0.0,
                    ),
                ),
                (
                    "model",
                    model,
                ),
            ]
        )

        pipeline.fit(
            X_fold_train,
            y_fold_train_model,
        )

        fold_predictions = (
            pipeline.predict(
                X_fold_valid
            )
        )

        if transform_target:
            fold_predictions = np.expm1(
                fold_predictions
            )

        metrics = regression_metrics(
            y_fold_valid_original,
            fold_predictions,
        )

        fold_results.append(
            {
                "experiment": experiment_name,
                "fold": fold_number,
                **metrics,
            }
        )

        print(
            f"{experiment_name} | "
            f"fold {fold_number} | "
            f"MAE={metrics['MAE']:.3f} | "
            f"RMSE={metrics['RMSE']:.3f} | "
            f"R2={metrics['R2']:.3f} | "
            f"MAPE={metrics['MAPE_percent']:.2f}%"
        )

    return pd.DataFrame(
        fold_results
    )

In [ ]:
experiment_A = (
    evaluate_xgboost_target_experiment(
        experiment_name="A_price_L1",
        objective="reg:absoluteerror",
        transform_target=False,
    )
)

experiment_B = (
    evaluate_xgboost_target_experiment(
        experiment_name="B_price_L2",
        objective="reg:squarederror",
        transform_target=False,
    )
)

experiment_C = (
    evaluate_xgboost_target_experiment(
        experiment_name="C_log1p",
        objective="reg:squarederror",
        transform_target=True,
    )
)


target_experiments = pd.concat(
    [
        experiment_A,
        experiment_B,
        experiment_C,
    ],
    ignore_index=True,
)


target_experiment_summary = (
    target_experiments
    .groupby("experiment")
    .agg(
        CV_R2_mean=(
            "R2",
            "mean",
        ),
        CV_R2_std=(
            "R2",
            "std",
        ),
        CV_RMSE_mean=(
            "RMSE",
            "mean",
        ),
        CV_RMSE_std=(
            "RMSE",
            "std",
        ),
        CV_MAE_mean=(
            "MAE",
            "mean",
        ),
        CV_MAE_std=(
            "MAE",
            "std",
        ),
        CV_MAPE_mean=(
            "MAPE_percent",
            "mean",
        ),
    )
    .sort_values(
        "CV_MAE_mean"
    )
    .reset_index()
)

display(
    target_experiment_summary
)

## 20. Selección y entrenamiento del modelo predictivo definitivo

A partir de la comparación de modelos, la optimización con Optuna y los experimentos sobre la variable objetivo, se selecciona como modelo final **XGBoost entrenado directamente sobre `price` con la función de pérdida `reg:absoluteerror`**.

La selección se basa principalmente en el MAE medio obtenido mediante validación cruzada espacial:

- LightGBM optimizado: MAE CV = 29.266 €
- XGBoost optimizado: MAE CV = 28.455 €

Además, entre las distintas configuraciones del objetivo, la alternativa `A_price_L1` obtuvo el menor MAE medio:

- `A_price_L1`: MAE CV = 28.455 €
- `C_log1p`: MAE CV = 28.900 €
- `B_price_L2`: MAE CV = 29.341 €

Por tanto, el modelo definitivo utiliza:

- XGBoost;
- `price` en escala original;
- `reg:absoluteerror`;
- los hiperparámetros seleccionados por Optuna;
- target encoding espacial H3;
- preprocesamiento e imputación integrados en el pipeline.

El modelo se entrena finalmente con la totalidad del conjunto de entrenamiento y se evalúa sobre el holdout espacial independiente del 20 %.

In [ ]:
# ============================================================
# PIPELINE XGBOOST DEFINITIVO
# ============================================================

best_params_xgb_final = (
    study_xgboost
    .best_params
    .copy()
)

h3_smoothing_final = (
    best_params_xgb_final
    .pop("h3_smoothing")
)

final_model = Pipeline(
    steps=[
        (
            "h3_target_encoder",
            H3TargetMeanEncoder(
                smoothing=h3_smoothing_final,
            ),
        ),
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "variance_filter",
            VarianceThreshold(
                threshold=0.0,
            ),
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:absoluteerror",
                **best_params_xgb_final,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
                verbosity=0,
            ),
        ),
    ]
)


# Entrenamiento definitivo
final_model.fit(
    X_train,
    y_train,
)


# Predicción sobre el holdout espacial
final_predictions = (
    final_model.predict(
        X_test
    )
)


# Métricas finales
final_test_metrics = (
    regression_metrics(
        y_test,
        final_predictions,
    )
)

print("=" * 70)
print("MODELO DEFINITIVO: XGBOOST")
print("=" * 70)

print(
    f"R2:   "
    f"{final_test_metrics['R2']:.6f}"
)

print(
    f"RMSE: "
    f"{final_test_metrics['RMSE']:.6f}"
)

print(
    f"MAE:  "
    f"{final_test_metrics['MAE']:.6f}"
)

print(
    f"MAPE: "
    f"{final_test_metrics['MAPE_percent']:.6f}%"
)

### 20.1. Evaluación final sobre el holdout espacial

El modelo definitivo se evalúa sobre el 20 % de observaciones reservado mediante separación espacial por celdas `h3_res8`.

Este conjunto no intervino en la optimización de hiperparámetros ni en la selección de la transformación de la variable objetivo.

Las métricas obtenidas representan por tanto la estimación final de rendimiento del sistema predictivo sobre zonas espaciales no utilizadas durante el entrenamiento.

In [ ]:
final_predictions_df = pd.DataFrame(
    {
        "y_true": y_test,
        "y_pred": final_predictions,
        "error": (
            final_predictions - y_test
        ),
        "absolute_error": np.abs(
            final_predictions - y_test
        ),
    }
)

display(
    final_predictions_df.head()
)

## 21. Guardado del modelo definitivo y resultados

El pipeline definitivo de XGBoost se guarda completo, incluyendo:

- target encoding H3;
- imputación;
- codificación de variables categóricas;
- filtrado de variables constantes;
- modelo XGBoost optimizado.

Además, se guardan los hiperparámetros seleccionados, la configuración de la partición espacial, las métricas finales y las predicciones obtenidas sobre el holdout del 20 %.

Esto permite reproducir posteriormente las predicciones sin volver a entrenar el modelo.

In [ ]:
joblib.dump(
    final_model,
    OUTPUT_XGBOOST_MODEL,
)


with OUTPUT_BEST_PARAMS.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        {
            "model": "XGBoost",
            "objective": "reg:absoluteerror",
            "target": "price",
            "target_transformation": None,

            "best_cv_mae": (
                study_xgboost.best_value
            ),

            "best_params": (
                study_xgboost.best_params
            ),

            "strata_column": STRATA_COLUMN,

            "holdout_group": "h3_res8",
            "cv_group": "h3_res8",

            "random_state": RANDOM_STATE,

            "holdout_test_ratio_target": (
                HOLDOUT_TEST_RATIO
            ),

            "holdout_test_ratio_observed": float(
                len(X_test) / len(X)
            ),

            "holdout_search_trials": (
                HOLDOUT_SEARCH_TRIALS
            ),

            "territorial_mean_difference": (
                mean_territorial_difference
            ),

            "territorial_max_difference": (
                max_territorial_difference
            ),

            "test_h3_res8_cells": sorted(
                map(
                    str,
                    test_cells,
                )
            ),

            "test_metrics": (
                final_test_metrics
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


final_test_comparison.to_csv(
    OUTPUT_METRICS,
    index=False,
    encoding="utf-8-sig",
)


test_prediction_table = pd.DataFrame(
    {
        "y_true": y_test,

        "y_pred": (
            final_predictions
        ),

        "residual": (
            y_test
            - final_predictions
        ),

        "absolute_error": np.abs(
            y_test
            - final_predictions
        ),

        "h3_res8": (
            groups_test_res8
        ),

        "territory": (
            strata_test
        ),
    }
)


test_prediction_table.to_csv(
    OUTPUT_TEST_PREDICTIONS,
    index=False,
    encoding="utf-8-sig",
)


print(
    f" Modelo guardado: "
    f"{OUTPUT_XGBOOST_MODEL}"
)

print(
    f" Métricas guardadas: "
    f"{OUTPUT_METRICS}"
)

print(
    f" Parámetros guardados: "
    f"{OUTPUT_BEST_PARAMS}"
)

print(
    f"Predicciones guardadas: "
    f"{OUTPUT_TEST_PREDICTIONS}"
)

## 22. Preparación de datos para SHAP

La interpretación del modelo definitivo se realiza mediante SHAP.

Los valores SHAP se calculan sobre la matriz que recibe directamente XGBoost, una vez aplicadas todas las transformaciones incluidas en el pipeline:

1. target encoding espacial H3;
2. imputación;
3. codificación OneHot de variables categóricas;
4. eliminación de variables constantes.

Como el modelo definitivo predice `price` directamente, los valores SHAP también se interpretan en la escala original del precio, sin necesidad de aplicar `expm1`.

In [ ]:
h3_transformer_fitted = (
    final_model
    .named_steps[
        "h3_target_encoder"
    ]
)

preprocessor_fitted = (
    final_model
    .named_steps[
        "preprocessor"
    ]
)

variance_filter_fitted = (
    final_model
    .named_steps[
        "variance_filter"
    ]
)

xgboost_model_fitted = (
    final_model
    .named_steps[
        "model"
    ]
)


X_test_h3 = (
    h3_transformer_fitted
    .transform(
        X_test
    )
)

X_test_preprocessed = (
    preprocessor_fitted
    .transform(
        X_test_h3
    )
)

X_test_model = (
    variance_filter_fitted
    .transform(
        X_test_preprocessed
    )
)


all_feature_names = (
    preprocessor_fitted
    .get_feature_names_out()
)

selected_feature_names = (
    all_feature_names[
        variance_filter_fitted
        .get_support()
    ]
)


print(
    "Features antes del filtro:",
    len(all_feature_names),
)

print(
    "Features utilizadas por XGBoost:",
    len(selected_feature_names),
)

## 23. Cálculo de valores SHAP

Para limitar el coste computacional se utiliza una muestra reproducible del conjunto de test.

`TreeExplainer` permite obtener la contribución de cada variable a las predicciones generadas por XGBoost.

In [ ]:
shap_sample_size = min(
    MAX_SHAP_ROWS,
    X_test_model.shape[0],
)


rng = np.random.default_rng(
    RANDOM_STATE
)


shap_sample_indices = rng.choice(
    X_test_model.shape[0],
    size=shap_sample_size,
    replace=False,
)


X_shap = X_test_model[
    shap_sample_indices
]


if hasattr(
    X_shap,
    "toarray",
):
    X_shap_dense = (
        X_shap.toarray()
    )
else:
    X_shap_dense = np.asarray(
        X_shap
    )


explainer = shap.TreeExplainer(
    xgboost_model_fitted
)


shap_values = explainer(
    X_shap_dense
)


shap_values.feature_names = (
    selected_feature_names
    .tolist()
)


print(
    "Valores SHAP calculados para",
    shap_sample_size,
    "alojamientos."
)

## 24. SHAP Beeswarm: importancia global

El gráfico beeswarm resume las variables que más influyen globalmente en las predicciones del modelo.

Los valores SHAP positivos incrementan la tarifa predicha, mientras que los negativos la reducen.

In [ ]:
plt.figure()

shap.plots.beeswarm(
    shap_values,
    max_display=25,
    show=False,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_SHAP_BEESWARM,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    f"Beeswarm guardado: "
    f"{OUTPUT_SHAP_BEESWARM}"
)

## 25. SHAP Waterfall: explicación individual

El gráfico waterfall permite interpretar una predicción concreta mostrando cómo cada variable desplaza el valor esperado del modelo hasta alcanzar la tarifa estimada para el alojamiento.

In [ ]:
SHAP_LOCAL_POSITION = 0

if not (
    0 <= SHAP_LOCAL_POSITION < shap_sample_size
):
    raise IndexError(
        "SHAP_LOCAL_POSITION está fuera del rango."
    )


plt.figure(
    figsize=(11, 10)
)

shap.plots.waterfall(
    shap_values[
        SHAP_LOCAL_POSITION
    ],
    max_display=20,
    show=False,
)

plt.gcf().set_size_inches(
    11,
    10,
)

plt.subplots_adjust(
    top=0.94,
    left=0.32,
    right=0.95,
    bottom=0.08,
)

plt.savefig(
    OUTPUT_SHAP_WATERFALL,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


original_test_position = int(
    shap_sample_indices[
        SHAP_LOCAL_POSITION
    ]
)

print(
    "Posición explicada dentro de X_test:",
    original_test_position,
)

print(
    "Precio real:",
    float(
        y_test.iloc[
            original_test_position
        ]
    ),
)

print(
    "Precio predicho:",
    float(
        final_predictions[
            original_test_position
        ]
    ),
)

print(
    f"Waterfall guardado: "
    f"{OUTPUT_SHAP_WATERFALL}"
)

## 26. Verificación del modelo guardado

Se vuelve a cargar el pipeline desde disco y se comprueba que reproduce exactamente las mismas predicciones que el modelo utilizado durante el entrenamiento.

In [ ]:
loaded_pipeline = joblib.load(
    OUTPUT_XGBOOST_MODEL
)


loaded_predictions = (
    loaded_pipeline.predict(
        X_test.head(10)
    )
)


original_predictions = (
    final_model.predict(
        X_test.head(10)
    )
)


assert np.allclose(
    loaded_predictions,
    original_predictions,
)


print(
    "El pipeline XGBoost guardado "
    "se carga correctamente y "
    "reproduce las predicciones."
)